# Depression and substance use in the NSDUH

A runnable companion to an analysis of the **National Survey on Drug Use and Health**,
2022 to 2024. Three years pooled give 134,897 adults with a usable depression outcome.

> ### This notebook runs on synthetic data
> The three public-use files total about 1.1 GB, far more than a notebook can carry. The
> generator below builds a dataset from this study's aggregate parameters, so every cell
> runs with no setup and the code paths are real.
>
> **Numbers produced here are simulated.** The real figures are on the
> [analysis page](https://johnfisher-ai.github.io/fisher-nsduh-analysis/analysis.html). Part 1b switches onto the real files once you have
> downloaded them from [SAMHSA](https://www.samhsa.gov/data/data-we-collect/nsduh-national-survey-drug-use-and-health).

The thing this notebook is really about: NSDUH is a **complex survey**, and treating it as
a simple random sample changes both the estimates and how precise they appear.

In [ ]:
import numpy as np, pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, cohen_kappa_score, confusion_matrix
import matplotlib.pyplot as plt
pd.set_option("display.width", 130)
print("ready")

## Part 1a. Synthetic data from the study's aggregates

In [ ]:
PARAMS = {
 "n": 134897,
 "years": [
  2022,
  2023,
  2024
 ],
 "marginals": {
  "ALC_PASTMO": 0.5265,
  "MJ_PASTMO": 0.1932,
  "CIG_PASTMO": 0.1434
 },
 "cat": {
  "IRSEX": {
   "2": 0.5529,
   "1": 0.4471
  },
  "CATAG6": {
   "2": 0.301,
   "4": 0.2707,
   "3": 0.2032,
   "6": 0.1134,
   "5": 0.1117
  },
  "INCOME": {
   "4": 0.4206,
   "2": 0.2644,
   "1": 0.1663,
   "3": 0.1487
  },
  "HEALTH": {
   "2": 0.3685,
   "3": 0.315,
   "1": 0.1924,
   "4": 0.1241
  }
 },
 "mde_logit": {
  "const": -3.3622,
  "ALC_PASTMO": 0.1252,
  "MJ_PASTMO": 0.5564,
  "CIG_PASTMO": 0.1558,
  "IRSEX": 0.5753,
  "CATAG6": -0.4675,
  "INCOME": -0.0013,
  "HEALTH": 0.6541
 },
 "weight_by_age": {
  "2": 827.2,
  "3": 1436.2,
  "4": 1704.7,
  "5": 3970.7,
  "6": 3700.8
 }
}

SEED = 1

def simulate(n=None, seed=SEED):
    """Build a dataset with the study's marginals, its logistic relationship to MDE, and a
    weight profile that varies by age, which is what makes weighting matter."""
    rng = np.random.default_rng(seed)
    n = n or PARAMS["n"]
    d = pd.DataFrame({c: rng.random(n) < p for c, p in PARAMS["marginals"].items()}).astype(int)
    for col, probs in PARAMS["cat"].items():
        lv = np.array([int(k) for k in probs]); pr = np.array(list(probs.values()), float)
        d[col] = rng.choice(lv, size=n, p=pr / pr.sum())
    b = PARAMS["mde_logit"]
    lin = b["const"] + sum(d[k] * v for k, v in b.items() if k != "const")
    d["MDE"] = (rng.random(n) < 1 / (1 + np.exp(-lin))).astype(int)
    # NSDUH over-samples the young, so their weights are smaller. That is the whole point.
    wmap = {int(k): v for k, v in PARAMS["weight_by_age"].items()}
    d["WT"] = d.CATAG6.map(wmap) * rng.lognormal(0, .25, n)
    d["YEAR"] = rng.choice(PARAMS["years"], n)
    return d

df = simulate()
print(f"simulated {len(df):,} adults")
print(f"unweighted MDE: {df.MDE.mean()*100:.2f}%   weighted: {np.average(df.MDE, weights=df.WT)*100:.2f}%")
df.head()

### 1b. Using the real files instead

Download the public-use files from [SAMHSA](https://www.samhsa.gov/data/data-we-collect/nsduh-national-survey-drug-use-and-health). Read only the columns you need: each
file has about 2,650, and reading all of them is slow and memory hungry.

In [ ]:
USE_REAL_DATA = False
DATA_DIR = "."

KEEP = ["IRSEX","CATAG6","INCOME","HEALTH2","IRALCRC","IRMJRC","CIGMON",
        "AMDEYR","AMIPY","ANALWT2_C","VESTR_C","VEREP"]

def load_real(data_dir=DATA_DIR, years=(2022, 2023, 2024)):
    frames = []
    for y in years:
        d = pd.read_csv(f"{data_dir}/NSDUH_{y}_Tab.txt", sep="\t",
                        usecols=lambda c: c in KEEP, low_memory=False)
        d["YEAR"] = y
        frames.append(d)
    d = pd.concat(frames, ignore_index=True)
    d["ALC_PASTMO"] = (d.IRALCRC == 1).astype(int)
    d["MJ_PASTMO"]  = (d.IRMJRC  == 1).astype(int)
    d["CIG_PASTMO"] = (d.CIGMON  == 1).astype(int)
    d["MDE"]    = d.AMDEYR.where(d.AMDEYR.isin([1, 2])).map({1: 1, 2: 0})
    d["HEALTH"] = d.HEALTH2.where(d.HEALTH2.isin([1, 2, 3, 4]))
    d["WT"] = d.ANALWT2_C / len(years)      # pooling: divide by the number of years
    return d[d.MDE.notna() & d.HEALTH.notna()].copy()

if USE_REAL_DATA:
    df = load_real()
    print(f"loaded {len(df):,} adults")
else:
    print("using synthetic data (set USE_REAL_DATA = True once the files are downloaded)")

## Part 2. The survey design

Every respondent carries a weight saying how many people they represent. Unweighted, you
describe the respondents; weighted, you estimate the population. When a survey deliberately
over-samples a group whose behaviour differs, the two answers differ.

In [ ]:
wm = lambda s, w: float(np.average(s, weights=w))

rows = []
for c in ["ALC_PASTMO", "MJ_PASTMO", "CIG_PASTMO", "MDE"]:
    rows.append(dict(measure=c,
                     unweighted=round(df[c].mean()*100, 2),
                     weighted=round(wm(df[c], df.WT)*100, 2)))
tab = pd.DataFrame(rows)
tab["difference"] = (tab.unweighted - tab.weighted).round(2)
print(tab.to_string(index=False))

print("\nBy age band, the two nearly agree within each band:")
for k in sorted(df.CATAG6.unique()):
    g = df[df.CATAG6 == k]
    print(f"  age code {k}: n={len(g):6,d}  unweighted {g.MDE.mean()*100:5.2f}%   "
          f"weighted {wm(g.MDE, g.WT)*100:5.2f}%")
print("\nIt is the age MIX that differs, which is why only the overall figure moves.")

## Part 3. Reduce, then group

Seven measures on different scales, so standardise first, then take principal components and
cluster on the leading three.

> **The synthetic data understates this step.** The generator draws each variable
> independently, so there is no correlation structure for PCA to find and the components come
> out near-equal at about 1/7 each. On the real data the first three carry
> 55.6%. The code below is the right code; only the simulated input
> is uninformative here. Run it against the real files to see the actual structure.

In [ ]:
VARS = ["ALC_PASTMO","MJ_PASTMO","CIG_PASTMO","IRSEX","CATAG6","INCOME","HEALTH"]
Xs = StandardScaler().fit_transform(df[VARS].astype(float))
pca = PCA(random_state=SEED).fit(Xs)
print("variance explained:", np.round(pca.explained_variance_ratio_*100, 1))
print("cumulative       :", np.round(np.cumsum(pca.explained_variance_ratio_)*100, 1))

scores = pca.transform(Xs)[:, :3]
km = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(scores)
df["cluster"] = km.labels_
print()
print(df.groupby("cluster")[["ALC_PASTMO","MJ_PASTMO","CIG_PASTMO","MDE"]].mean().mul(100).round(1))

**A caution.** K-means returns four groups whether or not four groups exist. Before
reading anything into them, check that the same structure comes back on resamples.

In [ ]:
rng = np.random.default_rng(SEED)
sizes = []
for _ in range(20):
    idx = rng.choice(len(scores), size=min(40000, len(scores)), replace=False)
    k2 = KMeans(n_clusters=4, n_init=5, random_state=int(rng.integers(1e6))).fit(scores[idx])
    sizes.append(np.sort(np.bincount(k2.labels_, minlength=4) / len(idx) * 100))
sizes = np.array(sizes)
print("cluster sizes over 20 resamples (sorted):")
print("  mean:", np.round(sizes.mean(axis=0), 1))
print("  sd  :", np.round(sizes.std(axis=0, ddof=1), 2))
print("\nReproducible is not the same as real: a continuum can be cut reproducibly too.")

## Part 4. Prediction, and why accuracy misleads

The outcome is rare. Watch what accuracy does about that.

In [ ]:
X = StandardScaler().fit_transform(df[VARS].astype(float))
y = df.MDE.astype(int).values
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
model = LogisticRegression(solver="liblinear", l1_ratio=1, C=1.0, max_iter=2000)
prob = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]

pred = (prob >= .5).astype(int)
print(f"AUC       {roc_auc_score(y, prob):.4f}")
print(f"accuracy  {(pred == y).mean():.4f}")
print(f"kappa     {cohen_kappa_score(y, pred):.4f}")
print(f"flagged   {pred.sum():,} of {len(y):,}   (true positives in data: {y.sum():,})")
print(f"imbalance {(y==0).sum()/(y==1).sum():.2f} to 1")
print("\nHigh accuracy, near-zero kappa: the model is mostly saying 'no' to everyone.")

In [ ]:
rows = []
for t in (0.50, 0.30, 0.20, 0.15, 0.10):
    pr = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pr).ravel()
    rows.append(dict(cutoff=t, sensitivity=round(tp/(tp+fn), 3),
                     specificity=round(tn/(tn+fp), 3),
                     precision=round(tp/(tp+fp), 3) if (tp+fp) else 0.0,
                     kappa=round(cohen_kappa_score(y, pr), 3),
                     flagged_pct=round(100*pr.mean(), 1)))
thr = pd.DataFrame(rows)
print(thr.to_string(index=False))
print("\nAUC is identical for every row: only the decision rule changed.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thr.cutoff, thr.sensitivity*100, "o-", label="Sensitivity", color="#2a78d6")
ax.plot(thr.cutoff, thr.flagged_pct, "o-", label="% flagged", color="#eb6834")
ax.set_xlabel("decision threshold"); ax.set_ylabel("percent"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("Lowering the cut-off finds more cases, at the cost of flagging more people")
plt.tight_layout(); plt.show()

## Part 5. What the design does to the standard errors

Respondents are drawn in clusters, so they carry less information than the same number of
independent draws. A model that ignores that reports standard errors which are too small.

In [ ]:
Xd = sm.add_constant(df[VARS].astype(float))
naive = sm.GLM(y, Xd, family=sm.families.Binomial()).fit()

# With the real data, cluster on the stratum-replicate combination:
#   groups = (df.VESTR_C.astype(str) + "_" + df.VEREP.astype(str)).values
# The synthetic data has no such structure, so age band stands in as a stub.
groups = df.CATAG6.values
design = sm.GLM(y, Xd, family=sm.families.Binomial(), var_weights=df.WT.values).fit(
    cov_type="cluster", cov_kwds={"groups": groups})

comp = pd.DataFrame({"naive_se": naive.bse, "design_se": design.bse})
comp["ratio"] = (comp.design_se / comp.naive_se).round(2)
print(comp.round(5).to_string())
print("\nOn the real data these ratios run about 1.7 to 2.0: naive intervals are roughly")
print("half the width they should be.")

## What the real analysis found

On the real files: 134,897 adults, MDE 8.44% weighted against
11.29% unweighted, AUC 0.7406,
accuracy 88.7% with kappa 0.0401, and design-adjusted
standard errors 1.89x to 1.96x
the naive ones.

Full write-up: [https://johnfisher-ai.github.io/fisher-nsduh-analysis/analysis.html](https://johnfisher-ai.github.io/fisher-nsduh-analysis/analysis.html)